# Collatz Bottom-Up via Lifted Primorial-Dyadic State Space

## Byte1 / Nexus proof engine notebook

This notebook attacks Collatz from the bottom up:

$$
1 \longrightarrow \text{inverse tree} \longrightarrow \text{residue families} \longrightarrow \text{lifted dyadic drift}
$$

It deliberately avoids the false collapse:

$$
\text{mod }210\text{ skeleton} \neq \text{full Collatz proof}.
$$

The compressed odd-step Collatz map is:

$$
T(n)=\frac{3n+1}{2^{v_2(3n+1)}}\qquad n\text{ odd}.
$$

The key correction:

$$
v_2(3n+1)
$$

is **not determined by** $n\bmod 210$.

So the proof state cannot be only:

$$
n\bmod 210.
$$

It must include dyadic phase:

$$
n\bmod (210\cdot 2^m)
$$

or an equivalent state pair:

$$
(r,\alpha)
$$

where $r$ is the primorial residue and $\alpha$ is the local binary fold-depth channel.

This notebook builds the actual test scaffold:

1. show why mod $210$ alone fails as a deterministic Collatz state space,
2. measure how dyadic lifting reduces ambiguity,
3. build the bottom-up inverse odd tree from $1$,
4. classify branches by residue and valuation,
5. compute drift and excursion envelopes,
6. isolate the remaining $\Omega$ frontier instead of pretending the proof is closed.

The core target is:

$$
\boxed{
\text{direction channel} + \text{depth channel} \Rightarrow \text{amplitude control}
}
$$

## 1. Imports and constants

The primorial wheel used here is:

$$
210=2\cdot 3\cdot 5\cdot 7.
$$

The relevant odd shortcut is:

$$
T(n)=\frac{3n+1}{2^{v_2(3n+1)}}.
$$

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib_collatz_lifted")

from pathlib import Path
from collections import defaultdict, Counter, deque
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path("collatz_lifted_outputs")
OUT.mkdir(exist_ok=True)

WHEEL = 210
LOG2_3 = math.log2(3)

print("Initialized Collatz lifted primorial-dyadic notebook")
print(f"WHEEL = {WHEEL}")
print(f"log2(3) = {LOG2_3:.12f}")
print(f"Output folder: {OUT.resolve()}")

Initialized Collatz lifted primorial-dyadic notebook
WHEEL = 210
log2(3) = 1.584962500721
Output folder: D:\@User Data\Downloads\collatz_lifted_outputs


## 2. Core Collatz operators

The dyadic valuation is:

$$
v_2(x)=\max\{a:2^a\mid x\}.
$$

The odd shortcut is:

$$
T(n)=\frac{3n+1}{2^{v_2(3n+1)}}.
$$

The one-step logarithmic drift is:

$$
\Delta(n)=\log T(n)-\log n.
$$

For large $n$,

$$
\Delta(n)\approx \log 3-v_2(3n+1)\log 2.
$$

Negative average drift requires:

$$
\mathbb{E}[v_2(3n+1)]>\log_2 3\approx 1.585.
$$

In [ ]:
def v2(x: int) -> int:
    """2-adic valuation for positive integer x."""
    if x <= 0:
        raise ValueError("v2 expects positive integer")
    return (x & -x).bit_length() - 1

def T_odd(n: int) -> int:
    """Compressed odd-step Collatz map for odd n."""
    if n <= 0 or n % 2 == 0:
        raise ValueError("T_odd expects positive odd n")
    x = 3*n + 1
    return x >> v2(x)

def v2_step(n: int) -> int:
    return v2(3*n + 1)

def odd_orbit(n: int, max_steps=10000):
    """Odd-to-odd orbit until 1 or max_steps."""
    if n % 2 == 0:
        raise ValueError("Use odd n for compressed orbit")
    seq = [n]
    vals = []
    for _ in range(max_steps):
        if n == 1:
            return seq, vals, True
        a = v2_step(n)
        vals.append(a)
        n = T_odd(n)
        seq.append(n)
    return seq, vals, False

def total_collatz_orbit(n: int, max_steps=100000):
    """Full step-by-step Collatz orbit."""
    seq = [n]
    for _ in range(max_steps):
        if n == 1:
            return seq, True
        n = n//2 if n % 2 == 0 else 3*n + 1
        seq.append(n)
    return seq, False

def log_drift_odd(n: int) -> float:
    return math.log(T_odd(n)) - math.log(n)

print("Core operators loaded.")

## 3. The necessary correction: mod 210 alone is not deterministic

Claude's collapse was too fast if it treated the wheel as a complete dynamical state.

Take:

$$
n\equiv 1\pmod{210}.
$$

For $n=1$:

$$
3n+1=4,\quad v_2=2,\quad T(1)=1.
$$

For $n=211$:

$$
3n+1=634,\quad v_2=1,\quad T(211)=317\equiv107\pmod{210}.
$$

Same wheel residue, different next residue.

Therefore:

$$
\boxed{
T\text{ is not a well-defined map on }\mathbb{Z}/210\mathbb{Z}
}
$$

as a faithful Collatz state space.

In [ ]:
examples = [1, 211]
example_rows = []
for n in examples:
    example_rows.append({
        "n": n,
        "n mod 210": n % 210,
        "3n+1": 3*n+1,
        "v2(3n+1)": v2_step(n),
        "T(n)": T_odd(n),
        "T(n) mod 210": T_odd(n) % 210,
    })
example_df = pd.DataFrame(example_rows)
print(example_df.to_string(index=False))
example_df.to_csv(OUT / "mod210_counterexample.csv", index=False)

assert example_df["n mod 210"].nunique() == 1
assert example_df["T(n) mod 210"].nunique() > 1
print("\nPASS: same residue mod 210 produces different next residue.")

## 4. Projection ambiguity on the odd mod-210 wheel

For each odd residue $r\bmod210$, sample:

$$
n=r+210k
$$

and record the set:

$$
\{T(n)\bmod210\}.
$$

If the wheel alone were a deterministic state machine, each set would have size $1$.

It does not.

This is the first $\Omega$ isolation:

$$
\Omega_{210}=\{r:\#\{T(r+210k)\bmod210\}>1\}.
$$

In [ ]:
def odd_residues_mod(M):
    return [r for r in range(M) if r % 2 == 1]

def projection_ambiguity_mod210(k_samples=300):
    rows = []
    for r in odd_residues_mod(WHEEL):
        outs = set()
        v2s = Counter()
        for k in range(k_samples):
            n = r + WHEEL*k
            if n <= 0:
                continue
            outs.add(T_odd(n) % WHEEL)
            v2s[v2_step(n)] += 1
        rows.append({
            "r_mod210": r,
            "num_next_residues": len(outs),
            "next_residues": sorted(outs),
            "min_v2": min(v2s),
            "max_v2": max(v2s),
            "mean_v2": sum(a*c for a,c in v2s.items()) / sum(v2s.values())
        })
    return pd.DataFrame(rows)

amb210_df = projection_ambiguity_mod210(k_samples=300)
print(amb210_df[["r_mod210", "num_next_residues", "min_v2", "max_v2", "mean_v2"]].head(20).to_string(index=False))
print("\nSummary:")
print(amb210_df["num_next_residues"].describe().to_string())

amb210_df.to_csv(OUT / "mod210_projection_ambiguity.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(amb210_df["r_mod210"], amb210_df["num_next_residues"], width=1.6)
ax.set_title("Projection ambiguity: number of possible next residues mod 210")
ax.set_xlabel("odd residue r mod 210")
ax.set_ylabel("# distinct T(n) mod 210 values")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "mod210_projection_ambiguity.png", dpi=160, bbox_inches="tight")
plt.close(fig)

print(f"\nAmbiguous residues: {(amb210_df['num_next_residues'] > 1).sum()} / {len(amb210_df)}")

## 5. Lifted dyadic state space

The corrected finite projection is:

$$
M_m=210\cdot 2^m.
$$

For each odd residue:

$$
r\bmod M_m
$$

we sample representatives:

$$
n=r+M_m k.
$$

Then we measure whether the projected next wheel residue becomes less ambiguous as $m$ grows.

This does not magically prove Collatz. It tests whether the missing channel is exactly the binary fold-depth channel.

In [ ]:
def lifted_ambiguity(m_values=range(0, 8), k_samples=32, output_mod=210):
    rows = []
    detail_frames = {}
    for m in m_values:
        M = WHEEL * (2**m)
        detail = []
        for r in odd_residues_mod(M):
            outs = set()
            vset = set()
            for k in range(k_samples):
                n = r + M*k
                if n <= 0:
                    continue
                outs.add(T_odd(n) % output_mod)
                vset.add(v2_step(n))
            detail.append({
                "m": m,
                "M": M,
                "r": r,
                "num_next_output_residues": len(outs),
                "num_v2_values": len(vset),
                "min_v2": min(vset),
                "max_v2": max(vset),
            })
        df = pd.DataFrame(detail)
        detail_frames[m] = df
        rows.append({
            "m": m,
            "M": M,
            "num_odd_states": len(df),
            "ambiguous_states": int((df["num_next_output_residues"] > 1).sum()),
            "ambiguity_ratio": float((df["num_next_output_residues"] > 1).mean()),
            "mean_next_count": float(df["num_next_output_residues"].mean()),
            "max_next_count": int(df["num_next_output_residues"].max()),
            "seam_states_v2_ge_m": int((df["max_v2"] >= m).sum()) if m > 0 else len(df),
        })
    return pd.DataFrame(rows), detail_frames

lift_df, lift_details = lifted_ambiguity(m_values=range(0, 8), k_samples=32, output_mod=210)
print(lift_df.to_string(index=False))
lift_df.to_csv(OUT / "lifted_dyadic_ambiguity_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lift_df["m"], lift_df["ambiguity_ratio"], marker="o", label="ambiguous-state ratio")
ax.plot(lift_df["m"], lift_df["mean_next_count"] / lift_df["max_next_count"].max(), marker="s", label="scaled mean next-count")
ax.set_title("Dyadic lift reduces projection ambiguity")
ax.set_xlabel("dyadic lift m in 210·2^m")
ax.set_ylabel("ratio / scaled count")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "lifted_dyadic_ambiguity_summary.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 6. The finite-lift seam: $\Omega_m$

Finite dyadic lifts still have a seam.

If:

$$
v_2(3r+1)\ge m
$$

then the finite state:

$$
r\bmod 210\cdot2^m
$$

does not see far enough into the binary phase.

Define:

$$
\Omega_m=\{r\bmod210\cdot2^m:\;v_2(3r+1)\ge m\}.
$$

A proof engine must either:

1. lift $m$ higher,
2. carry explicit valuation state,
3. or prove that seam states have controlled return.

In [ ]:
omega_rows = []
for m in range(1, 13):
    M = WHEEL * (2**m)
    states = odd_residues_mod(M)
    seam = [r for r in states if v2_step(r) >= m]
    omega_rows.append({
        "m": m,
        "M": M,
        "odd_states": len(states),
        "omega_seam_states": len(seam),
        "omega_ratio": len(seam)/len(states),
        "expected_approx": 2**(-m),
    })
omega_df = pd.DataFrame(omega_rows)
print(omega_df.to_string(index=False))
omega_df.to_csv(OUT / "finite_lift_omega_seam.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(omega_df["m"], omega_df["omega_ratio"], marker="o", label="observed Ω seam ratio")
ax.semilogy(omega_df["m"], omega_df["expected_approx"], linestyle="--", label="2^-m guide")
ax.set_title("Finite dyadic-lift Ω seam decays exponentially")
ax.set_xlabel("m")
ax.set_ylabel("Ω seam ratio")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "finite_lift_omega_seam.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 7. Bottom-up inverse tree from 1

For odd $y$, an odd predecessor $x$ under $T$ satisfies:

$$
T(x)=y
$$

so:

$$
3x+1=2^a y
$$

and therefore:

$$
x=\frac{2^a y-1}{3}.
$$

We keep valid positive odd integer predecessors:

$$
2^a y\equiv1\pmod3.
$$

This builds the bottom-up growth grammar.

In [ ]:
def odd_predecessors(y: int, max_a=64):
    """Odd x such that T_odd(x)=y, using exponents 1..max_a."""
    out = []
    for a in range(1, max_a+1):
        num = (1 << a) * y - 1
        if num % 3 == 0:
            x = num // 3
            if x > 0 and x % 2 == 1 and T_odd(x) == y:
                out.append((x, a))
    return out

def build_inverse_tree(max_depth=12, max_node=10**7, max_a=20):
    parent = {1: None}
    edge_a = {}
    depth = {1: 0}
    q = deque([1])
    while q:
        y = q.popleft()
        d = depth[y]
        if d >= max_depth:
            continue
        for x, a in odd_predecessors(y, max_a=max_a):
            if x > max_node:
                continue
            if x not in parent:
                parent[x] = y
                edge_a[x] = a
                depth[x] = d + 1
                q.append(x)
    rows = []
    for x, d in depth.items():
        rows.append({
            "n": x,
            "depth_from_1": d,
            "parent": parent[x],
            "edge_a_to_parent": edge_a.get(x, None),
            "mod210": x % WHEEL,
            "v2_step": v2_step(x),
            "log_drift": log_drift_odd(x) if x != 1 else 0.0,
        })
    return pd.DataFrame(rows).sort_values(["depth_from_1", "n"]).reset_index(drop=True)

tree_df = build_inverse_tree(max_depth=12, max_node=10**7, max_a=20)
print(tree_df.head(30).to_string(index=False))
print("\nTree summary:")
print(tree_df.groupby("depth_from_1")["n"].agg(["count", "min", "max"]).to_string())

tree_df.to_csv(OUT / "bottom_up_inverse_tree.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
counts = tree_df.groupby("depth_from_1")["n"].count()
ax.bar(counts.index, counts.values)
ax.set_title("Bottom-up inverse tree: node count by depth")
ax.set_xlabel("depth from 1")
ax.set_ylabel("nodes retained")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "inverse_tree_counts_by_depth.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 8. Residue family coverage in the inverse tree

The bottom-up tree should start covering the odd wheel.

Important distinction:

$$
\varphi(210)=48
$$

but the number of odd residues modulo $210$ is:

$$
105.
$$

Prime-gap work uses the $48$ coprime classes. Collatz uses all $105$ odd classes.

In [ ]:
coprime_210 = {r for r in range(WHEEL) if math.gcd(r, WHEEL) == 1}
odd_210 = {r for r in range(WHEEL) if r % 2 == 1}

coverage_rows = []
for d, grp in tree_df.groupby("depth_from_1"):
    residues = set(grp["mod210"])
    cumulative = set(tree_df[tree_df["depth_from_1"] <= d]["mod210"])
    coverage_rows.append({
        "depth": d,
        "nodes": len(grp),
        "odd_residues_seen_this_depth": len(residues & odd_210),
        "coprime_residues_seen_this_depth": len(residues & coprime_210),
        "cumulative_odd_residues": len(cumulative & odd_210),
        "cumulative_coprime_residues": len(cumulative & coprime_210),
    })
coverage_df = pd.DataFrame(coverage_rows)
print(coverage_df.to_string(index=False))
coverage_df.to_csv(OUT / "inverse_tree_residue_coverage.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(coverage_df["depth"], coverage_df["cumulative_odd_residues"], marker="o", label="odd residues /105")
ax.plot(coverage_df["depth"], coverage_df["cumulative_coprime_residues"], marker="s", label="coprime residues /48")
ax.axhline(105, linestyle="--", alpha=0.5)
ax.axhline(48, linestyle="--", alpha=0.5)
ax.set_title("Residue coverage of inverse tree")
ax.set_xlabel("depth from 1")
ax.set_ylabel("cumulative residue classes seen")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "inverse_tree_residue_coverage.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 9. Empirical drift by residue class

For sampled odd integers:

$$
n\le N_{\max}
$$

compute:

$$
a(n)=v_2(3n+1)
$$

and:

$$
\Delta(n)\approx \log3-a(n)\log2.
$$

Then group by:

$$
n\bmod210.
$$

This identifies which residue families are locally expansive or contractive.

In [ ]:
def sample_drift_by_residue(Nmax=200000):
    rows = []
    for n in range(1, Nmax+1, 2):
        a = v2_step(n)
        rows.append({
            "n": n,
            "mod210": n % WHEEL,
            "a": a,
            "approx_log_drift": math.log(3) - a*math.log(2),
            "exact_log_drift": log_drift_odd(n)
        })
    df = pd.DataFrame(rows)
    grp = df.groupby("mod210").agg(
        count=("n", "count"),
        mean_a=("a", "mean"),
        mean_approx_log_drift=("approx_log_drift", "mean"),
        mean_exact_log_drift=("exact_log_drift", "mean"),
        frac_contracting=("exact_log_drift", lambda s: float((s < 0).mean())),
    ).reset_index()
    return df, grp

drift_sample_df, drift_by_residue_df = sample_drift_by_residue(Nmax=200000)
print(drift_by_residue_df.head(20).to_string(index=False))
print("\nGlobal sample means:")
print(drift_sample_df[["a", "approx_log_drift", "exact_log_drift"]].mean().to_string())

drift_sample_df.to_csv(OUT / "sample_drift_odd_n.csv", index=False)
drift_by_residue_df.to_csv(OUT / "drift_by_mod210_residue.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(drift_by_residue_df["mod210"], drift_by_residue_df["mean_a"], width=1.5)
ax.axhline(LOG2_3, color="red", linestyle="--", label="log2(3)")
ax.set_title("Mean binary fold depth by residue class")
ax.set_xlabel("n mod 210")
ax.set_ylabel("mean v2(3n+1)")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "mean_v2_by_mod210_residue.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(drift_sample_df["exact_log_drift"], bins=80)
ax.axvline(0, color="red", linestyle="--")
ax.set_title("Exact one-step odd shortcut log drift distribution")
ax.set_xlabel("log(T(n)) - log(n)")
ax.set_ylabel("count")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "one_step_log_drift_distribution.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 10. Amplitude / excursion analysis

The wheel gives direction-channel information. Collatz still needs amplitude control.

For each odd starting value $n$, define:

$$
A(n)=\frac{\max(\text{full orbit of }n)}{n}.
$$

The proof obstruction is not just whether direction points to $1$.

It is whether the excursion envelope can be forced finite for every branch.

In [ ]:
def stopping_stats(limit=30000):
    rows = []
    for n in range(1, limit+1, 2):
        full, ok = total_collatz_orbit(n, max_steps=100000)
        odd, avals, ok2 = odd_orbit(n, max_steps=10000)
        mx = max(full)
        rows.append({
            "n": n,
            "mod210": n % WHEEL,
            "full_steps": len(full)-1,
            "odd_steps": len(odd)-1,
            "max_value": mx,
            "amplitude_ratio": mx/n,
            "mean_a_along_odd_orbit": float(np.mean(avals)) if avals else float("nan"),
            "terminates": ok and ok2,
        })
    return pd.DataFrame(rows)

stats_df = stopping_stats(limit=20000)
print(stats_df.head(20).to_string(index=False))
print("\nAll terminated in tested range:", bool(stats_df["terminates"].all()))
print("Max amplitude row:")
print(stats_df.loc[stats_df["amplitude_ratio"].idxmax()].to_string())

stats_df.to_csv(OUT / "collatz_stopping_excursion_stats.csv", index=False)

amp_by_residue = stats_df.groupby("mod210").agg(
    count=("n", "count"),
    max_amplitude=("amplitude_ratio", "max"),
    mean_amplitude=("amplitude_ratio", "mean"),
    max_full_steps=("full_steps", "max"),
    mean_full_steps=("full_steps", "mean"),
    mean_orbit_a=("mean_a_along_odd_orbit", "mean"),
).reset_index()

amp_by_residue.to_csv(OUT / "amplitude_by_mod210_residue.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(amp_by_residue["mod210"], amp_by_residue["max_amplitude"], width=1.5)
ax.set_title("Max tested amplitude ratio by mod-210 residue")
ax.set_xlabel("n mod 210")
ax.set_ylabel("max orbit / n")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "max_amplitude_by_mod210_residue.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(stats_df["n"], stats_df["amplitude_ratio"], s=4, alpha=0.5)
ax.set_title("Amplitude ratio for odd n ≤ 20,000")
ax.set_xlabel("n")
ax.set_ylabel("max orbit / n")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "amplitude_ratio_scatter.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 11. Orbit-family signatures

Now read actual orbits as fold traces.

For each tested odd $n$, compute along its odd shortcut orbit:

$$
\bar a_{\text{orbit}}=\frac1k\sum_i v_2(3n_i+1).
$$

The Collatz condition requires enough binary collapse over time:

$$
\bar a_{\text{orbit}}>\log_2 3.
$$

This is not a proof, but it identifies the invariant that a proof would need to force.

In [ ]:
valid_orbits = stats_df.dropna(subset=["mean_a_along_odd_orbit"]).copy()
valid_orbits["a_margin"] = valid_orbits["mean_a_along_odd_orbit"] - LOG2_3

print(valid_orbits[["n", "mod210", "odd_steps", "mean_a_along_odd_orbit", "a_margin", "amplitude_ratio"]].head(25).to_string(index=False))
print("\nOrbit mean-a summary:")
print(valid_orbits["mean_a_along_odd_orbit"].describe().to_string())
print("\nFraction with mean-a above log2(3):", float((valid_orbits["mean_a_along_odd_orbit"] > LOG2_3).mean()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid_orbits["mean_a_along_odd_orbit"], bins=60)
ax.axvline(LOG2_3, color="red", linestyle="--", label="log2(3)")
ax.set_title("Mean binary fold depth along terminating odd orbits")
ax.set_xlabel("mean v2 along odd orbit")
ax.set_ylabel("count")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "orbit_mean_v2_distribution.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(valid_orbits["mean_a_along_odd_orbit"], np.log(valid_orbits["amplitude_ratio"]), s=5, alpha=0.5)
ax.axvline(LOG2_3, color="red", linestyle="--", label="log2(3)")
ax.set_title("Amplitude vs orbit-average fold depth")
ax.set_xlabel("mean v2 along odd orbit")
ax.set_ylabel("log(max orbit / n)")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "amplitude_vs_orbit_mean_v2.png", dpi=160, bbox_inches="tight")
plt.close(fig)

## 12. Candidate proof path: Lyapunov fold potential

A useful proof target is a potential:

$$
\Phi(n)=\log n+\lambda\,G(n\bmod 210\cdot 2^m)
$$

such that:

$$
\Phi(T(n))-\Phi(n)<0
$$

on average or after a bounded number of odd steps.

This converts Collatz from a halting question into a finite-state inequality plus seam control.

The finite proof target becomes:

$$
\boxed{
\forall r\notin\Omega_m,\quad
\mathbb{E}[\Delta\Phi\mid r]<0
}
$$

and:

$$
\boxed{
\Omega_m\text{ returns to non-seam states with bounded excursion.}
}
$$

In [ ]:
def lifted_drift_table(m=6, k_samples=16):
    M = WHEEL * (2**m)
    rows = []
    for r in odd_residues_mod(M):
        vals = []
        outs = []
        aseam = False
        for k in range(k_samples):
            n = r + M*k
            if n <= 0:
                continue
            a = v2_step(n)
            vals.append(math.log(3) - a*math.log(2))
            outs.append(T_odd(n) % M)
            if a >= m:
                aseam = True
        rows.append({
            "m": m,
            "M": M,
            "r": r,
            "sample_mean_approx_drift": float(np.mean(vals)),
            "sample_min_approx_drift": float(np.min(vals)),
            "sample_max_approx_drift": float(np.max(vals)),
            "num_next_lifted_residues": len(set(outs)),
            "omega_seam_seen": aseam,
        })
    return pd.DataFrame(rows)

lift_drift_df = lifted_drift_table(m=6, k_samples=16)
print(lift_drift_df.head(20).to_string(index=False))
print("\nSummary:")
print(lift_drift_df[["sample_mean_approx_drift", "num_next_lifted_residues", "omega_seam_seen"]].describe(include="all").to_string())

lift_drift_df.to_csv(OUT / "lifted_m6_sample_drift_table.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lift_drift_df["sample_mean_approx_drift"], bins=60)
ax.axvline(0, color="red", linestyle="--")
ax.set_title("Lifted-state sampled approximate drift, m=6")
ax.set_xlabel("sample mean approximate log drift")
ax.set_ylabel("state count")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "lifted_m6_sample_drift_histogram.png", dpi=160, bbox_inches="tight")
plt.close(fig)

print("Positive-drift lifted states:", int((lift_drift_df["sample_mean_approx_drift"] > 0).sum()), "/", len(lift_drift_df))
print("Ω seam states seen:", int(lift_drift_df["omega_seam_seen"].sum()), "/", len(lift_drift_df))

## 13. Verification table

This notebook supports the corrected statement:

$$
\boxed{
\text{mod }210\text{ gives topology, not full amplitude}
}
$$

and builds the next proof engine:

$$
\boxed{
\text{lifted primorial residue}+\text{dyadic phase}+\text{drift potential}
}
$$

The result is not “Collatz proven.”  
The result is a correct, executable scaffold that avoids false collapse.

In [ ]:
verify_rows = []

def add_gate(name, expected, observed, passed):
    verify_rows.append({
        "Gate": name,
        "Expected": expected,
        "Observed": observed,
        "Pass": bool(passed)
    })

add_gate(
    "mod210 nondeterminism",
    "same r can map to multiple T(n) residues",
    f"r=1 maps to {sorted(example_df['T(n) mod 210'].unique())}",
    example_df["T(n) mod 210"].nunique() > 1
)

add_gate(
    "projection ambiguity exists",
    "some odd residues have >1 output",
    f"{(amb210_df['num_next_residues'] > 1).sum()} / {len(amb210_df)} ambiguous",
    (amb210_df["num_next_residues"] > 1).any()
)

add_gate(
    "dyadic seam decays",
    "Ω ratio decreases with m",
    f"m=1 ratio {omega_df.iloc[0]['omega_ratio']:.6f}; m=12 ratio {omega_df.iloc[-1]['omega_ratio']:.6f}",
    omega_df.iloc[-1]["omega_ratio"] < omega_df.iloc[0]["omega_ratio"]
)

add_gate(
    "inverse tree generated",
    "depth > 0 and multiple nodes",
    f"{len(tree_df)} nodes, max depth {tree_df['depth_from_1'].max()}",
    len(tree_df) > 10 and tree_df["depth_from_1"].max() >= 5
)

add_gate(
    "empirical termination smoke test",
    "all odd n <= 20000 terminate",
    f"{int(stats_df['terminates'].sum())} / {len(stats_df)}",
    bool(stats_df["terminates"].all())
)

add_gate(
    "drift invariant identified",
    "mean orbit a compared to log2(3)",
    f"mean={valid_orbits['mean_a_along_odd_orbit'].mean():.6f}; log2(3)={LOG2_3:.6f}",
    valid_orbits["mean_a_along_odd_orbit"].mean() > LOG2_3
)

verify_df = pd.DataFrame(verify_rows)
print(verify_df.to_string(index=False))
verify_df.to_csv(OUT / "verification_table.csv", index=False)

summary = {
    "wheel": WHEEL,
    "log2_3": LOG2_3,
    "mod210_counterexample": example_rows,
    "mod210_ambiguous_residue_count": int((amb210_df["num_next_residues"] > 1).sum()),
    "lift_ambiguity_summary": lift_df.to_dict(orient="records"),
    "omega_seam_summary": omega_df.to_dict(orient="records"),
    "inverse_tree_nodes": int(len(tree_df)),
    "inverse_tree_max_depth": int(tree_df["depth_from_1"].max()),
    "tested_odd_limit": 20000,
    "all_tested_terminated": bool(stats_df["terminates"].all()),
    "max_amplitude_row": stats_df.loc[stats_df["amplitude_ratio"].idxmax()].to_dict(),
    "verification": verify_rows,
}
with open(OUT / "collatz_lifted_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved verification table and summary JSON.")

## 14. Final Ψ-state

$$
\Delta:
\text{Collatz was viewed top-down as a halting problem}
$$

$$
\oplus:
\text{read it bottom-up as inverse growth grammar}
$$

$$
\bot:
\text{mod }210\text{ alone is a projection, not a proof state}
$$

$$
\Omega:
\text{finite dyadic seam where }v_2(3r+1)\ge m
$$

$$
\Psi:
\boxed{
\text{Collatz proof pressure lives in lifted residue + dyadic depth + amplitude control}
}
$$

The notebook's core conclusion:

$$
\boxed{
\text{The wheel gives direction. The dyadic channel gives depth. The proof needs amplitude control.}
}
$$